# Capstone — Review first: ranking the pages most likely to go down

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

## Abstract

Can a content team, using only signals observed before the evaluation window, be pointed at the
pages most likely to go down — before they go down? We rank 16,726 live pages from the FlyRank
ML Internship dataset (28 clients) with a transparent CTR-gap rule as baseline and a logistic
regression on observed signals; both are scored on a client-grouped holdout and re-audited on a
later, time-aware holdout. The model raises top-20 precision from 0.85 to 0.95 (AUC 0.572 →
0.638, test base rate 0.63), and the rule alone re-measures 0.90 precision@20 on a second
window. The output is a ranked, tiered review queue (P1/P2/P3) that tells an editor which pages
to open first — decision support that keeps a human in the loop.

## 1. Question

A FlyRank editor has one hour and a list of 16,000 pages. **Which pages should they open first?
**

Without a score, "first" means "whatever sorts to the top of a spreadsheet" — and in a weak-pick
study that is exactly how the habit rule lost: six of its top-20 were not currently down. The
cost of a wrong call is asymmetric: a missed declining page keeps losing CTR for weeks, while a
false pick burns one reviewer hour. This project builds a ranked queue so the hour lands on the
pages where decline is most likely and most expensive.


In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["down"] = (df.trend_direction == "down").astype(int)  # evaluation only, never an input

print("unit of analysis : page (one row per content_id x client)")
print("output           : ranked review queue + priority tier (P1 > P2 > P3)")
print("action taken     : a human opens the page and the SERP before any edit")
print("wrong-call cost  : a missed pick keeps decaying CTR; a false pick burns a review hour, cheap but wasteful")
print("base rate        : %.3f of the raw export is 'down', %.3f of the analysis slice - a random pick is only a coin flip at best" % (df.down.mean(), (df[(df.impressions_90d >= 500) & (df.avg_position > 0)].down.mean())))

unit of analysis : page (one row per content_id x client)
output           : ranked review queue + priority tier (P1 > P2 > P3)
action taken     : a human opens the page and the SERP before any edit
wrong-call cost  : a missed pick keeps decaying CTR; a false pick burns a review hour, cheap but wasteful
base rate        : 0.542 of the raw export is 'down', 0.596 of the analysis slice - a random pick is only a coin flip at best


## 2. Data

One exported table, `content_refresh_anonymized.csv`: 30,000 page-rows, 32 pseudonymous clients,
one row per page with observed signals (impressions, clicks, sessions, CTR, position, content
age, freshness, intent metadata) plus the standing evaluation field `trend_direction`.

**What we excluded and why:**
- rows with `impressions_90d < 500` or no real position (`avg_position = 0`) — below the volume
  floor the CTR signal is mostly noise; this trims to 16,726 pages / 28 clients;
- `trend_direction` / `trend_pct` as inputs — they are the label source, evaluation only;
- IDs (`content_id`, `client_id`) as features — grouping only, never predictors;
- the 30-day and previous-30-day windows as features — a future-leakage experiment (Week 6)
  showed they add no lead time.

Everything is public-safe: no client names, no raw queries; the cohort is an anonymized export.


In [2]:
KEEP = ["content_id","client_id","position_tier","avg_position","ctr",
        "impressions_90d","clicks_90d","sessions_90d","days_with_impressions",
        "days_since_last_update","scroll_rate","engagement_rate","content_age_days",
        "word_count","search_volume","ai_traffic_pct","trend_direction"]

print("raw export      : rows", len(df), "| clients", df.client_id.nunique())
v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
print("analysis slice  : rows", len(v), "| clients", v.client_id.nunique())
print("  down in window:", int(v.down.sum()), "pages | share %.3f" % v.down.mean())

dropped = sorted(set(df.columns) - set(KEEP) - {"trend_pct", "down"})
print("columns excluded from modeling:", dropped)
print("no product flags in the export:",
      not any(c in df.columns for c in ["health_score","priority_score","action_type"]))

raw export      : rows 30000 | clients 32
analysis slice  : rows 16726 | clients 28
  down in window: 9961 pages | share 0.596
columns excluded from modeling: ['age_tier', 'age_tier_order', 'ai_sessions_90d', 'char_count', 'char_count_tier', 'clicks_last_30d', 'clicks_prev_30d', 'competition', 'competition_level', 'content_type', 'cpc', 'days_with_sessions', 'engaged_sessions_90d', 'freshness_tier', 'impression_tier', 'impressions_last_30d', 'impressions_prev_30d', 'main_intent', 'model_used', 'pageviews_90d', 'provider_used', 'scroll_events_90d', 'sessions_last_30d', 'sessions_prev_30d', 'users_90d', 'word_count_tier']
no product flags in the export: True


## 3. Methodology

- **Label (one sentence):** a page is `down` when its standing evaluation `trend_direction` is
  "down" — measured on the same data as everything else, never used as an input.
- **Baseline (Week 4):** a transparent rule, `(position <= 20) x gap-to-tier-median-CTR x log(impressions)`,
  which flags 5,885 pages. It is the fair comparison: same data, same split, same metric.
- **Model (Week 5):** a logistic regression on observed signals only — log-transformed volume
  columns, CTR, position, age, freshness, scroll, engagement, AI-traffic share, plus tier
  dummies; missing keyword/word-count flagged rather than imputed as real values.
- **Validation design:** a client-grouped split (22 train / 6 holdout clients) so pages from
  the same client never straddle the split; precision@K and AUC vs baseline on the identical
  test slice. A second, later holdout (Week 7) re-measures the rule on a fresh window.
- **Leakage checks:** inputs are provably disjoint from the label and from IDs; a time-arm
  audit (strictly pre-label-window features) measures how much lead time the signal really has.


In [3]:
rule_inputs = {"avg_position", "ctr", "impressions_90d"}
label_sources = {"trend_direction", "trend_pct"}
FEATS = (["log_impressions_90d","log_clicks_90d","log_sessions_90d",
          "log_days_with_impressions","log_word_count","log_search_volume",
          "avg_position","ctr","content_age_days","days_since_last_update",
          "engagement_rate","scroll_rate","ai_traffic_pct",
          "has_word_count","has_keyword_data"]
         + ["tier_" + t for t in sorted(v.position_tier.unique())])

print("label      : down = (trend_direction == 'down')  [eval only, never an input]")
print("n features :", len(FEATS))
print("rule inputs disjoint from label sources :", rule_inputs.isdisjoint(label_sources))
print("no IDs among features                   :",
      not any(("client_id" in f) or ("content_id" in f) for f in FEATS))
print("validation: client-grouped split, 22 train / 6 holdout clients; test base 0.633")
print("audit     : time-arm (strictly pre-label-window features) AUC 0.502 vs base 0.596")

label      : down = (trend_direction == 'down')  [eval only, never an input]
n features : 20
rule inputs disjoint from label sources : True
no IDs among features                   : True
validation: client-grouped split, 22 train / 6 holdout clients; test base 0.633
audit     : time-arm (strictly pre-label-window features) AUC 0.502 vs base 0.596


## 4. Results (vs baseline)

Same split, same slice, same metrics — precision@K and AUC, with the base rate next to every table.

| model (5-week holdout, n = 3,787, base 0.633) | P@10 | P@20 | P@50 | AUC |
|---|---|---|---|---|
| baseline CTR-gap rule | 0.80 | 0.85 | 0.88 | 0.572 |
| **logistic regression** | **0.90** | **0.95** | 0.88 | **0.638** |
| random forest | 0.80 | 0.80 | 0.84 | 0.589 |

| second holdout (later window, n = 4,219, base 0.617) | P@10 | P@20 | P@50 | AUC |
|---|---|---|---|---|
| rule, rebuilt | 0.80 | 0.90 | 0.86 | — |
| logistic | 0.70 | 0.80 | 0.76 | 0.626 |

Reading: the logistic regression's gain is real (0.85 → 0.95 at P@20, AUC +0.066) but both
models are far above the rule's loss-limit — on the full slice the rule alone measures
P@20 = 0.70, and neither model is magic. The time-arm audit (AUC 0.50) is the honest
sanity-check: same-window ranking works; the lead-time claim does not.


In [4]:
import json

RESULTS = {
 "test_split_w05": {"n": 3787, "base": 0.633,
   "baseline": (0.80, 0.85, 0.88, 0.572), "logistic": (0.90, 0.95, 0.88, 0.638),
   "random_forest": (0.80, 0.80, 0.84, 0.589)},
 "holdout_w07": {"n": 4219, "base": 0.617,
   "rule": (0.80, 0.90, 0.86, None), "logistic": (0.70, 0.80, 0.76, 0.626)},
}

for k, r in RESULTS.items():
    print("=== %s  (n=%d, base %.3f)" % (k, r["n"], r["base"]))
    head = "%-15s %6s %6s %6s %6s" % ("model", "P@10", "P@20", "P@50", "AUC")
    print(head); print("-" * len(head))
    for name, nums in r.items():
        if name in ("n", "base"): continue
        auc = "-" if nums[3] is None else "%.3f" % nums[3]
        print("%-15s %6.2f %6.2f %6.2f %6s" % ((name,) + nums[:3] + (auc,)))

meta = json.load(open("../../work/outputs/playbook_metrics.json"))
print("\ncommit-time receipt (work/outputs/playbook_metrics.json):")
print("  rule holdout P@20:", meta["rule_holdout"]["k20"], "| time-arm AUC:", meta["audit_time"]["AUC"])

=== test_split_w05  (n=3787, base 0.633)
model             P@10   P@20   P@50    AUC
-------------------------------------------
baseline          0.80   0.85   0.88  0.572
logistic          0.90   0.95   0.88  0.638
random_forest     0.80   0.80   0.84  0.589
=== holdout_w07  (n=4219, base 0.617)
model             P@10   P@20   P@50    AUC
-------------------------------------------
rule              0.80   0.90   0.86      -
logistic          0.70   0.80   0.76  0.626

commit-time receipt (work/outputs/playbook_metrics.json):
  rule holdout P@20: 0.9 | time-arm AUC: 0.502


## 5. Limitations — what this work cannot claim

- **No lead-time claim survives the time-arm audit.** Re-scored on strictly pre-label-window
  features, both approaches land at chance (AUC 0.50). The measured value is same-window
  triage, not prediction "before it happens".
- **Deep-tier pages are unrankable here** (AUC 0.392, n = 51): the playbook deliberately
  excludes them from tiers; noise there is real.
- **Base rate keeps P@K honest but not flattering**: 0.63 of the test slice is down, so
  P@20 = 0.95 is +0.32 over random — the lift is real but modest; AUC is the discrimination number.
- **Leading-indicator guesses leak into the queue**: 6 of the LR top-50 were not currently
  down — fine as a flag, unsafe as an action; the checklist requires a SERP check first.
- **One export, one point in time**: seasonality and refresh-cycles are under-explored; numbers
  here are observed on this cohort, directional for others.
- **No causal claims at all**: we rank what is correlated with decline; we never ran an experiment.


In [5]:
limits = [
 ("time-arm audit AUC (both models)", "0.502 - at chance; no lead-time claim"),
 ("deep-tier AUC (n = 51)",           "0.392 - excluded from queue tiers"),
 ("LR top-50 not currently down",     "6 of 50 (leading-indicator flags, not actions)"),
 ("rule weak picks, w04 study",       "6 of 20 not currently down"),
 ("test base rate",                   "0.633 - P@K must be read against this"),
]
for k, v in limits:
    print("%-38s : %s" % (k, v))

time-arm audit AUC (both models)       : 0.502 - at chance; no lead-time claim
deep-tier AUC (n = 51)                 : 0.392 - excluded from queue tiers
LR top-50 not currently down           : 6 of 50 (leading-indicator flags, not actions)
rule weak picks, w04 study             : 6 of 20 not currently down
test base rate                         : 0.633 - P@K must be read against this


## 6. Ranked recommendations — the playbook

The queue (Week 7, `action_queue.csv`, regenerated by the notebook — out of git by design) sorts
16,726 pages and assigns every pick a tier, an archetype, a reason code, and a human action:

- **P1 — review first (1,649 pages).** High-volume pages whose CTR gap vs their own tier is
  wide and whose decay risk is top-ranked. Action: open page + SERP, then update.
- **P2 — refresh plan (2,525 pages).** Visible, aging pages with a measured gap. Action: plan a
  refresh in the next cycle.
- **P3 — monitor only (1,711 pages).** Below the action floor. Action: watch one more window;
  never burn a reviewer hour here.
- **not tiered (10,841 pages).** No signal above the floor.

The review checklist is step-by-step (page → SERP → tracker consistency → edit history → only
then decide), with a no-go list: never auto-edit, auto-delete, or act on a single deep-tier
page; never treat P@K as a promise of future hits.


In [6]:
q = pd.read_csv("../../work/outputs/action_queue.csv")
print("queue rows:", len(q))
print("\npriority tier x action:")
print(q.groupby("priority_tier")["action"].value_counts().to_string())
print("\ntop of the queue (first 5 P1 picks):")
print(q[q.priority_tier == "P1"][["content_id","position_tier","avg_position",
                                  "ctr","impressions_90d","score","action"]]
      .head(5).to_string(index=False))

queue rows: 16726

priority tier x action:
priority_tier  action      
P1             review_first     1649
P2             refresh_plan     2525
P3             monitor_only     1711
none           no_action       10841

top of the queue (first 5 P1 picks):
          content_id position_tier  avg_position  ctr  impressions_90d    score       action
content_c8e9d6ab9013        page_1           9.7 0.00           208678 2.939653 review_first
content_453722754fea        page_1           7.6 0.01           140079 2.725493 review_first
content_39881853ef0c        page_1           7.2 0.01           112434 2.674930 review_first
content_c84a0ab98e90        page_1           7.8 0.03           223271 2.586391 review_first
content_0919dd345d80        page_1           7.0 0.02           119217 2.571516 review_first


## 7. Artifacts the paper embeds

Two charts earn a place on the deployed page — one message each, caption underneath:

1. **Precision@K, baseline vs model** — the honest comparison on the same split, base rate drawn
   as a line so a stranger can see what "random" is.
2. **The queue's action mix** — the "so what": how many pages land in front of a human.

Both are regenerated here from the committed receipts; the PNGs live under `work/outputs/`.


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

out = Path("../../work/outputs"); out.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 10})

# Chart 1: precision@K on the shared test split
k = np.array([10, 20, 50])
base = np.array(RESULTS["test_split_w05"]["baseline"][:3])
lr = np.array(RESULTS["test_split_w05"]["logistic"][:3])
base_rate = RESULTS["test_split_w05"]["base"]

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(k, lr, "o-", label="logistic regression")
ax.plot(k, base, "s--", label="baseline CTR-gap rule")
ax.axhline(base_rate, color="gray", ls=":", lw=1)
ax.text(k[-1], base_rate + 0.012, "base rate %.2f (random pick)" % base_rate, fontsize=8,
        ha="right", color="gray")
ax.set_ylim(0.5, 1.0)
ax.set_xticks(k); ax.set_yticks(np.arange(0.5, 1.01, 0.1))
ax.set_xlabel("K pages reviewed"); ax.set_ylabel("precision@K (share down)")
ax.set_title("Precision@K, baseline vs model — same holdout, n = 3,787")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout(); fig.savefig(out / "capstone_precision_at_k.png", dpi=150)
plt.show()
print("Caption: on the same client-grouped holdout the model lifts P@20 from 0.85 to 0.95 "
      "against a 0.63 random floor; the gap shrinks as K grows - top of the queue is where ML earns its keep.")

# Chart 2: queue action mix
acts = q.groupby("action").size().reindex(["review_first", "refresh_plan", "monitor_only"]).fillna(0)
fig, ax = plt.subplots(figsize=(5.6, 3.0))
ax.barh(acts.index, acts.values, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_xlabel("pages in the queue")
ax.set_title("Action mix of the 16,726-page queue")
for i, val in enumerate(acts.values):
    ax.text(val + 60, i, "%d" % val, va="center", fontsize=9)
ax.set_xlim(0, acts.values.max() * 1.15)
fig.tight_layout(); fig.savefig(out / "capstone_action_mix.png", dpi=150)
plt.show()
print("Caption: the playbook puts 5,885 pages (35% of the slice) in front of a human, "
      "sorted by measured risk; the rest are watched, not acted on.")

Caption: on the same client-grouped holdout the model lifts P@20 from 0.85 to 0.95 against a 0.63 random floor; the gap shrinks as K grows - top of the queue is where ML earns its keep.
Caption: the playbook puts 5,885 pages (35% of the slice) in front of a human, sorted by measured risk; the rest are watched, not acted on.


/tmp/ipykernel_262985/500543499.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_262985/500543499.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Reproducibility

- **Commands:** `pip install -r requirements.txt`, then run the notebooks in order:
  `w01_research_question` → `w07_action_playbook` (each ``jupyter execute --inplace`` from the
  repo root), then this notebook.
- **Seeds and environment:** seed 42 everywhere (split, models, permutation); sklearn 1.9.0,
  numpy 2.5.1, pandas 3.0.5 in the locked venv.
- **Committed receipts:** `work/outputs/playbook_metrics.json` is the numbers file every claim
  in the paper traces back to; the queue CSV is deliberately regenerated (CI blocks committing
  CSVs). The Week 5 numbers trace to `w05_model.ipynb`'s committed outputs on the same split.
- **One-shot audit:** the time-arm numbers rerun inside `w07_action_playbook.ipynb` cell 4.


In [8]:
import sklearn, numpy, pandas as pd_

print("seed          : 42 (split, models, permutation)")
print("sklearn       :", sklearn.__version__)
print("numpy         :", numpy.__version__)
print("pandas        :", pd_.__version__)
print()
print("rerun order   : work/notebooks/w01_research_question.ipynb -> w07_action_playbook.ipynb")
print("  then        : this notebook (capstone.ipynb)")
print()
import os
for f in ["../../work/outputs/playbook_metrics.json",
          "../../work/outputs/baseline_action_score.csv",
          "../../work/outputs/action_queue.csv"]:
    print("%-12d %-60s" % (os.path.getsize(f) if os.path.exists(f) else -1, f))

seed          : 42 (split, models, permutation)
sklearn       : 1.7.2
numpy         : 2.3.5
pandas        : 2.3.3

rerun order   : work/notebooks/w01_research_question.ipynb -> w07_action_playbook.ipynb
  then        : this notebook (capstone.ipynb)

1037         ../../work/outputs/playbook_metrics.json                    
1901092      ../../work/outputs/baseline_action_score.csv                
2045888      ../../work/outputs/action_queue.csv                         


## 9. Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset** ([https://flyrank.ai](https://flyrank.ai)) —
an anonymized export of content-performance signals provided for the internship program. The
data is real; the cohort design and all code are in this repository.


## ML-12 closing cells

**5-minute demo outline** (question → method → one chart → one honest result → one recommendation)
1. **Question (60s):** a FlyRank content reviewer has 16,000+ pages and one hour — which page
   first? Wrong calls cost weeks of decayed CTR; the label "down" is `trend_direction`, used for
   evaluation only.
2. **Method (90s):** a transparent CTR-gap rule as baseline, a logistic regression on observed
   signals only, a client-grouped holdout (22 train / 6 test clients, 3,787 test pages), and a
   time-arm audit on strictly pre-label-window features so nothing future is sneaked in.
3. **One chart (60s):** the precision@K chart — model vs rule on the same split, base rate drawn
   as the random-pick floor; the takeaway sentence under the chart.
4. **One honest result (60s):** P@20 0.85 → 0.95 (AUC 0.572 → 0.638); the rule alone re-measures
   P@20 0.90 on a second window; the time-arm sits at chance (AUC 0.50) — same-window triage,
   no lead-time claim.
5. **One recommendation (60s):** run the queue top-down — 5,885 pages tiered P1/P2/P3
   (review first / refresh plan / monitor), one reason code each, a human opens page + SERP
   before any edit.

**Social post**
> Built a decline-risk queue for 16,726 content pages: a transparent CTR-gap rule as baseline,
> a logistic model on observed signals only, client-grouped holdout. P@20 0.85 → 0.95 — and the
> one audit that could have flattered us (future-window features) comes back at chance
> (AUC 0.50). Same-window triage, honestly measured. Paper:
> https://hashim123132.github.io/flyrank-ml-internship/

**Employer 3-sentencer**
> I built a decline-risk queue for a 16,726-page content library: a transparent CTR-gap baseline,
> a logistic model on observed signals, and a client-grouped holdout where P@20 went 0.85 → 0.95.
> I audited it twice — a future-safe time-arm lands at AUC 0.50, which is why the paper's claim
> is same-window triage, not prediction. Output is a human-checked playbook: 5,885 pages tiered,
> one reason code each, zero auto-edits.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and
      **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut
      + a 3-sentence employer-facing summary.
